# 0 - Helpers

In [ ]:
from generate_time_comparison import *

# 1 - Retrieve networks

In [ ]:
scenarios_baseline = [
    "baseline",
    "baseline-rps",
    "baseline-co2-price25",
    "baseline-co2-price50",
    "baseline-co2-price100"
]

scenarios_25 = [
    "baseline",
    "energy-match-25",
    "hourly-match-25-90",
    "hourly-match-25-95",
    "hourly-match-25-98",
    "hourly-match-25-99",
]

scenarios_noadd = [
    "baseline",
    "hourly-match-noadd-10-99",
    "hourly-match-noadd-50-99",
    "hourly-match-noadd-90-99",
]

scenarios_sensitivity_25 = [
    "hourly-match-25-99",
    "hourly-match-EU-25-99",
    "hourly-match-no-LDES-25-99",
    "hourly-match-no-clean-firm-25-99"
]

scenarios_sensitivity_co2_25 = [
    "hourly-match-25-99",
    "hourly-match-co2-price25-25-99",
    "hourly-match-co2-price50-25-99",
    "hourly-match-co2-price100-25-99"
]

years = [2025, 2030]
# Main scenarios
scenarios_50 = [
    "baseline",
    "energy-match-50",
    "hourly-match-50-90",
    "hourly-match-50-95",
    "hourly-match-50-98",
    "hourly-match-50-99",
]

scenarios_sensitivity_50 = [
    "hourly-match-50-99",
    "hourly-match-EU-50-99",
    "hourly-match-no-LDES-50-99",
    "hourly-match-no-clean-firm-50-99",
]

scenarios_sensitivity_co2_50 = [
    "hourly-match-50-99",
    "hourly-match-co2-price25-50-99",
    "hourly-match-co2-price50-50-99",
]

# For all scenarios with 4 timesteps
years = [2025, 2030, 2035, 2040]
scenarios_all = {
    "baseline": scenarios_baseline,
    "main_ci_25": scenarios_25,
    "sensitivity_noadd": scenarios_noadd,
    "sensitivity_tech_ci_25": scenarios_sensitivity_25,
    "sensitivity_co2_ci_25": scenarios_sensitivity_co2_25,  
}

# For all scenarios with 2 timesteps
# years = [2025, 2030]
# scenarios_all = {
#     "main_ci_50": scenarios_50,
#     "sensitivity_tech_ci_50": scenarios_sensitivity_50,
#     "sensitivity_co2_ci_50": scenarios_sensitivity_co2_50,  
# }

df_networks_all = {}

for group, scenarios in scenarios_all.items():

    # Build MultiIndex
    index = pd.MultiIndex.from_product(
        [years, scenarios],
        names=["year", "scenario"]
    )

    # Create empty DataFrame
    df_networks = pd.DataFrame(index=index, columns=["network"])

    # Fill it
    for year, sc in index:
        try:
            n = pypsa.Network(f"../results/{sc}/networks/base_s_39___{year}.nc")
        except:
            print(f"{sc}-{year} not availabe")
            continue
        n = prepare_network(n)
        n.name = f"{sc}-{year}"
        df_networks.loc[(year, sc), "network"] = n

        m = strip_network_GoO(n)
        m.name = "GoO-" + m.name
        df_networks.loc[(year, sc), "GoO"] = m

    df_networks = df_networks.dropna()
    df_networks_all[group] = df_networks

# Calculate figsize automatically based on number of years
figsize_bar_resource_utilization = FIGSIZE_RESOURCE_UTILIZATION  # Fixed size for resource utilization bar plot
print(f"Using figsize for resource utilization bar plot: {figsize_bar_resource_utilization}")

# Combine all networks from different groups into a single DataFrame
df_networks_all = pd.concat([df for df in df_networks_all.values()], axis=0)
df_networks_all = df_networks_all[~df_networks_all.index.duplicated(keep='first')]

# 1 - Resource utilization

In [ ]:
figures_to_generate = [
        'n', # derive_resource_utilization
]

for scenario in df_networks_all.index.get_level_values("scenario").unique():
    fig_path_system = f"figures/resource_utilization/{scenario}"

    df_networks = df_networks_all.loc[[(year, scenario) for year in years]]
    
    _ , _ = derive_all_figures(
        df_networks,
        country=None,
        plot_fig=True, 
        save_fig=True, 
        fig_path=fig_path_system, 
        save_csv=False,
        figures=figures_to_generate,
        figsize_bar_resource_utilization=figsize_bar_resource_utilization
    )
